# Filtered GRU Remote Suite

Google Drive에 프로젝트와 학습용 filtered dataset을 옮겨둔 뒤, Colab에서 Drive를 마운트하고 그 경로에서 바로 학습을 실행하는 노트북이다.

이 노트북은 Colab에서 기존 dataset을 생성하거나 수정하지 않는다. 아래 파일이 이미 존재한다고 가정한다.

- `dataset/final_dataset_filtered.csv` (`label`, `label_3class` 포함)

실험 버전:

- `v30_binary_segment`: `final_dataset_filtered.csv`, `label`, positive=`1`, `segment_max`
- `v31_binary_last`: `final_dataset_filtered.csv`, `label`, positive=`1`, `last_frame`
- `v32_classes_fall_any`: `final_dataset_filtered.csv`, `label_3class`, positive=`1,2`, `segment_max`
- `v33_classes_falling_only`: `final_dataset_filtered.csv`, `label_3class`, positive=`1`, `last_frame`

각 버전은 같은 모델 후보(`gru_64_32`, `gru_96_48`, `gru_128_64`, `gru_64_32_light`)와 같은 시각화/STM32 export 파이프라인을 사용한다.

In [ ]:
# Runtime setup: use the project already copied to Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime

# 사용자가 Drive에 옮겨둔 프로젝트 위치 후보. 필요하면 여기에 직접 경로를 추가하세요.
PROJECT_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Graduate-Project/Falling-Model-Development'),
    Path('/content/drive/MyDrive/졸업 과제/Falling-Detection-Development'),
    Path('/content/drive/MyDrive/Falling-Model-Development'),
]

# 필요 시 직접 지정. None이면 위 후보에서 자동 탐색한다.
PROJECT_ROOT_OVERRIDE = None  # 예: Path('/content/drive/MyDrive/내폴더/Falling-Model-Development')

# Drive에 있는 repo를 최신 원격 브랜치로 맞추고 싶을 때만 True.
UPDATE_FROM_GIT = False
BRANCH = 'codex/filtered-gru-models'

if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE)
else:
    PROJECT_ROOT = next((p for p in PROJECT_ROOT_CANDIDATES if (p / 'scripts').exists()), None)

if PROJECT_ROOT is None or not (PROJECT_ROOT / 'scripts').exists():
    raise FileNotFoundError(
        'Drive에 Falling-Model-Development 프로젝트가 없습니다. '
        '로컬 프로젝트 폴더를 Google Drive로 옮긴 뒤 PROJECT_ROOT_CANDIDATES 또는 '
        'PROJECT_ROOT_OVERRIDE를 수정하세요.'
    )

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

if UPDATE_FROM_GIT:
    if not (PROJECT_ROOT / '.git').exists():
        raise FileNotFoundError('UPDATE_FROM_GIT=True 이지만 PROJECT_ROOT에 .git이 없습니다.')
    subprocess.run(['git', 'fetch', 'origin'], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('UPDATE_FROM_GIT =', UPDATE_FROM_GIT)
print('timestamp =', datetime.now().isoformat(timespec='seconds'))
print('dataset dir =', PROJECT_ROOT / 'dataset')


In [ ]:
# Dependencies
# Colab already includes most packages, but pin nothing here so the notebook remains portable.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'matplotlib', 'scikit-learn', 'pandas', 'numpy'
], check=True)

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

In [ ]:
# Experiment controls
RUN_VERSIONS = ['all']  # examples: ['v30_binary_segment'], ['v30_binary_segment', 'v31_binary_last'], ['all']
MODELS = 'all'          # or 'gru_64_32,gru_96_48'
EPOCHS = 40
BATCH_SIZE = 64
MIN_VAL_RECALL = 0.86
QUANT_EVAL_MAX_WINDOWS = 5000
SKIP_STM32_EXPORT = False

DATASET_DIR = PROJECT_ROOT / 'dataset'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'filtered_gru_remote_suite'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = {
    'v30_binary_segment': {
        'csv': DATASET_DIR / 'final_dataset_filtered.csv',
        'label_column': 'label',
        'positive_labels': '1',
        'label_mode': 'segment_max',
        'description': '2-class label, segment_max window label',
    },
    'v31_binary_last': {
        'csv': DATASET_DIR / 'final_dataset_filtered.csv',
        'label_column': 'label',
        'positive_labels': '1',
        'label_mode': 'last_frame',
        'description': '2-class label, last_frame window label',
    },
    'v32_classes_fall_any': {
        'csv': DATASET_DIR / 'final_dataset_filtered.csv',
        'label_column': 'label_3class',
        'positive_labels': '1,2',
        'label_mode': 'segment_max',
        'description': 'single filtered dataset mapped falling+fallen to binary fall',
    },
    'v33_classes_falling_only': {
        'csv': DATASET_DIR / 'final_dataset_filtered.csv',
        'label_column': 'label_3class',
        'positive_labels': '1',
        'label_mode': 'last_frame',
        'description': 'single filtered dataset mapped only falling to binary fall',
    },
}

selected_versions = list(EXPERIMENTS) if 'all' in RUN_VERSIONS else RUN_VERSIONS
display(pd.DataFrame([
    {'version': key, **{k: str(v) for k, v in cfg.items()}}
    for key, cfg in EXPERIMENTS.items()
    if key in selected_versions
]))


In [ ]:
# Validate required datasets without modifying them
def validate_dataset_for_experiment(version: str, cfg: dict):
    csv_path = Path(cfg['csv'])
    if not csv_path.exists():
        raise FileNotFoundError(f'Required dataset not found for {version}: {csv_path}')
    header = pd.read_csv(csv_path, nrows=0).columns.tolist()
    required = {'video_id', 'frame', 'time_sec', cfg['label_column']}
    missing = sorted(required - set(header))
    if missing:
        raise ValueError(f'{version} dataset is missing columns {missing}: {csv_path}')
    feature_count = sum(1 for col in header if col.startswith('kp')) + sum(1 for col in ['HSSC_y', 'HSSC_x', 'RWHC', 'VHSSC', 'AHSSC', 'AHSSC_x'] if col in header)
    print(f"[OK] {version}: {csv_path.name} label_column={cfg['label_column']} feature_count={feature_count}")

for version in selected_versions:
    validate_dataset_for_experiment(version, EXPERIMENTS[version])


In [ ]:
# Run remote training jobs sequentially
run_records = []

for version in selected_versions:
    cfg = EXPERIMENTS[version]
    output_dir = ARTIFACT_DIR / version
    cmd = [
        sys.executable, 'scripts/train_filtered_gru_suite.py',
        '--csv-path', str(cfg['csv']),
        '--output-dir', str(output_dir),
        '--label-column', cfg['label_column'],
        '--positive-labels', cfg['positive_labels'],
        '--label-mode', cfg['label_mode'],
        '--models', MODELS,
        '--epochs', str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--min-val-recall', str(MIN_VAL_RECALL),
        '--quant-eval-max-windows', str(QUANT_EVAL_MAX_WINDOWS),
    ]
    if SKIP_STM32_EXPORT:
        cmd.append('--skip-stm32-export')

    print('\n' + '=' * 100)
    print(f'RUN {version}: {cfg["description"]}')
    print(' '.join(cmd))
    print('=' * 100)
    subprocess.run(cmd, check=True)
    run_records.append({'version': version, 'output_dir': str(output_dir), **cfg})

pd.DataFrame(run_records).to_csv(ARTIFACT_DIR / 'remote_run_records.csv', index=False)
display(pd.DataFrame(run_records))

In [ ]:
# Aggregate version/model comparisons
frames = []
for version in selected_versions:
    comparison_path = ARTIFACT_DIR / version / 'model_comparison.csv'
    if comparison_path.exists():
        df = pd.read_csv(comparison_path)
        df.insert(0, 'version', version)
        frames.append(df)

if not frames:
    raise FileNotFoundError('No model_comparison.csv files found.')

summary = pd.concat(frames, ignore_index=True)
summary_path = ARTIFACT_DIR / 'all_versions_model_comparison.csv'
summary.to_csv(summary_path, index=False)

metric_cols = [c for c in ['test_accuracy', 'test_precision', 'test_recall', 'test_macro_f1', 'int8_accuracy', 'int8_macro_f1'] if c in summary.columns]
display(summary[['version', 'model', 'threshold', 'min_consecutive'] + metric_cols].sort_values('test_macro_f1', ascending=False))

plot_df = summary.copy()
plot_df['version_model'] = plot_df['version'] + '\n' + plot_df['model']
ax = plot_df.set_index('version_model')[['test_accuracy', 'test_macro_f1']].plot(kind='bar', figsize=(16, 5))
ax.axhline(0.90, color='crimson', linestyle='--', linewidth=1.2)
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Filtered GRU remote suite summary')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
summary_png = ARTIFACT_DIR / 'all_versions_summary.png'
plt.savefig(summary_png, dpi=160)
plt.show()

print('summary csv:', summary_path)
print('summary png:', summary_png)

In [ ]:
# Display generated plots inline for quick review
PLOT_FILES = [
    ('model_comparison.png', 'Version model comparison'),
    ('history.png', 'Training history'),
    ('confusion_matrix_test.png', 'Test confusion matrix'),
    ('roc_pr_test.png', 'Test ROC / PR curves'),
    ('stm32_runtime_comparison.png', 'STM32 runtime comparison'),
]

def display_plot_if_exists(path: Path, title: str):
    if path.exists():
        display(Markdown(f'#### {title}'))
        print(path)
        display(Image(filename=str(path)))
        return True
    return False

for version in selected_versions:
    version_dir = ARTIFACT_DIR / version
    display(Markdown(f'## {version}'))
    display_plot_if_exists(version_dir / 'model_comparison.png', 'Version model comparison')

    model_dirs = sorted(p for p in version_dir.iterdir() if p.is_dir()) if version_dir.exists() else []
    for model_dir in model_dirs:
        display(Markdown(f'### {model_dir.name}'))
        shown_any = False
        for filename, title in PLOT_FILES[1:]:
            shown_any = display_plot_if_exists(model_dir / filename, title) or shown_any
        if not shown_any:
            print('No plot images found in', model_dir)
